# <center> Feature Engineering </center>

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")

from src.features.iv import calcular_iv, iv_contagem, classificar_iv

In [ ]:
# LOADING THE DATASET
df = pd.read_csv("../data/processed/credit_processed.csv")
print(f"Dataset carregado: {df.shape}")

In [ ]:
# CREATING BINARY FEATURES FOR LATE PAYEMENTS

df['teve_atraso_90dias'] = (df["NumberOfTimes90DaysLate"] > 0).astype(int)
df['teve_atraso_60dias'] = (df['NumberOfTime60-89DaysPastDueNotWorse'] > 0).astype(int)
df['teve_atraso_30dias'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] > 0).astype(int)

In [ ]:
# CHECKING
for col in ['teve_atraso_90dias', 'teve_atraso_60dias', 'teve_atraso_30dias']:
    taxa = df.groupby(col)['SeriousDlqin2yrs'].mean()
    print(f"\n{col}:")
    print(taxa.round(3))

In [ ]:
# CREATING A FEATURE CALLED "TOTAL_ATRASOS" (SOMANDO OS ATRASOS DE 30, 60 E 90 DIAS)
df['total_atrasos'] = (df['NumberOfTimes90DaysLate'] + df['NumberOfTime60-89DaysPastDueNotWorse'] + df['NumberOfTime30-59DaysPastDueNotWorse'])

In [ ]:
# CREATING A FEATURE FOR ANY TYPE OF LATE PAYMENT
df['teve_qualquer_atraso'] = (df['total_atrasos'] > 0).astype(int)

In [ ]:
# CHECKING THE DISTRIBUITION
print("\n=== total_atrasos — distribuição ===")
print(df["total_atrasos"].describe().round(2))
print(f"\nTaxa inadimplência — sem atraso: "
      f"{df[df['teve_qualquer_atraso']==0]['SeriousDlqin2yrs'].mean():.2%}")
print(f"Taxa inadimplência — com atraso: "
      f"{df[df['teve_qualquer_atraso']==1]['SeriousDlqin2yrs'].mean():.2%}")

In [ ]:
# CREATING FEATURES FOR REASON AND INTENSITY

#INCOME PER CAPITA
df['renda_per_capita'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

# TOTAL MORTGAGE CREDIT EXPOSURE VS. TOTAL
df['proporcao_credito_imobiliario'] = np.where(
    df['NumberOfOpenCreditLinesAndLoans'] > 0,
    df['NumberRealEstateLoansOrLines'] / df['NumberOfOpenCreditLinesAndLoans'],0
)

print("\n --- Renda Per Capita - Estatisticas ---")
print(df['renda_per_capita'].describe().round(2))

In [ ]:
# SIMPLE RISK SCORE (INTERACTION FEATURES)

# Normalizing each component to 0 or 1, before combining

def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

In [ ]:
# APPLYING THE FUNCTION

df['score_risco'] = (
    0.5 * normalizar(df['RevolvingUtilizationOfUnsecuredLines']) +
    0.3 * normalizar(df['NumberOfTimes90DaysLate']) +
    0.2 * normalizar(df['total_atrasos'])
)

print("\n--- Score de Risco - Estatisticas ---")
print(df['score_risco'].describe().round(4))
print(f"\nCorrelação entre score de risco e inadimplência: "
      f"{df['score_risco'].corr(df['SeriousDlqin2yrs']):.4f}")

In [ ]:
# VALIDATING THE PREDICTIVE POWER OF NEW FEATURES

# Validação de IV — features originais vs features criadas
print("=== IV features binárias criadas ===")
for col in ["teve_atraso_90dias", "teve_atraso_60dias",
            "teve_atraso_30dias", "teve_qualquer_atraso"]:
    iv = calcular_iv(df, col, "SeriousDlqin2yrs", bins=2)
    print(f"{col}: {iv:.4f} — {classificar_iv(iv)}")

print("\n=== IV features de razão criadas ===")
for col in ["renda_per_capita", "proporcao_credito_imobiliario", "score_risco"]:
    iv = calcular_iv(df, col, "SeriousDlqin2yrs")
    print(f"{col}: {iv:.4f} — {classificar_iv(iv)}")

print("\n=== Comparação original vs criada ===")
comparacao = {
    "NumberOfTimes90DaysLate (original)": iv_contagem(df, "NumberOfTimes90DaysLate", "SeriousDlqin2yrs"),
    "teve_atraso_90dias (criada)": calcular_iv(df, "teve_atraso_90dias", "SeriousDlqin2yrs", bins=2),
    "MonthlyIncome (original)": calcular_iv(df, "MonthlyIncome", "SeriousDlqin2yrs"),
    "renda_per_capita (criada)": calcular_iv(df, "renda_per_capita", "SeriousDlqin2yrs"),
}
for feature, iv in comparacao.items():
    print(f"{feature}: {iv:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DECISÃO FINAL DE FEATURES — CRITÉRIOS E JUSTIFICATIVAS
# ══════════════════════════════════════════════════════════════════════════════
#
# Critérios de seleção aplicados (em ordem de prioridade):
#   1. Poder preditivo (IV > 0.02)
#   2. Disponibilidade em produção (feature existe no momento da concessão)
#   3. Ausência de redundância severa (correlação > 0.7 entre features)
#   4. Interpretabilidade de negócio (explicável para o comitê de crédito)
#
# ──────────────────────────────────────────────────────────────────────────────
# FEATURES MANTIDAS
# ──────────────────────────────────────────────────────────────────────────────
#
# [IV 1.11] RevolvingUtilizationOfUnsecuredLines
#   → Feature mais preditiva do dataset. Representa o comportamento atual
#     do cliente com crédito rotativo. Padrão monotônico confirmado na EDA.
#     Sem evidência de leakage — é uma medida do momento da análise.
#
# [IV 0.88] NumberOfTimes90DaysLate
#   → Sinal mais grave de comportamento histórico. WoE mostrou que um único
#     atraso de 90 dias multiplica o risco por 9x. Mantida a contagem original
#     pois captura a severidade (1x vs 3x atrasos), não apenas a ocorrência.
#
# [IV 0.84] teve_atraso_90dias (criada)
#   → Binarização de NumberOfTimes90DaysLate. Complementa a contagem original:
#     enquanto a contagem captura severidade, o binário captura a ocorrência
#     com sinal mais limpo. Modelos lineares se beneficiam do binário;
#     modelos de árvore extraem o melhor das duas representações.
#
# [IV 1.18] teve_qualquer_atraso (criada)
#   → Agrega os três tipos de atraso em um único sinal binário. Captura
#     o perfil geral de inadimplente histórico independente da janela de tempo.
#     IV alto é herdado da combinação de três features fortes — não é leakage.
#
# [IV 0.76] NumberOfTime30-59DaysPastDueNotWorse
#   → Atrasos leves têm dinâmica diferente dos atrasos graves. Um cliente
#     pode ter muitos atrasos de 30 dias sem nunca chegar a 90 dias —
#     perfil de risco distinto que o modelo precisa aprender separadamente.
#
# [IV 0.60] NumberOfTime60-89DaysPastDueNotWorse
#   → Janela intermediária de atraso. Complementa as outras duas janelas
#     para o modelo capturar a trajetória de deterioração do cliente.
#
# [IV 0.75] total_atrasos (criada)
#   → Soma das três janelas de atraso. Captura a exposição histórica total
#     ao risco, independente da janela. Feature de intensidade agregada.
#
# [IV 1.11] score_risco (criada)
#   → Índice composto ponderado pelos sinais mais fortes (RevolvingUtilization,
#     NumberOfTimes90DaysLate, total_atrasos). Entregamos ao modelo um sinal
#     pré-processado — ele decide se agrega valor além das features individuais.
#
# [IV 0.26] age
#   → Única feature demográfica com poder preditivo forte. Relação monotônica
#     confirmada na EDA: inadimplência cai consistentemente com a idade.
#     Não correlacionada com as features de comportamento — adiciona dimensão
#     independente ao modelo.
#
# [IV 0.097] renda_per_capita (criada)
#   → Supera MonthlyIncome isolado (IV 0.067) ao incorporar o número de
#     dependentes. Renda de R$5.000 para uma pessoa é diferente de R$5.000
#     para uma família de 5. Feature de negócio com justificativa clara.
#
# [IV 0.067] MonthlyIncome
#   → Mantida junto com renda_per_capita pois captura a capacidade absoluta
#     de pagamento, enquanto renda_per_capita captura a relativa. O modelo
#     pode extrair sinais distintos das duas representações.
#
# [IV 0.067] NumberOfOpenCreditLinesAndLoans
#   → Exposição total ao sistema de crédito. Cliente com muitas linhas abertas
#     tem maior risco de comprometimento de renda — sinal independente das
#     features de atraso.
#
# [IV 0.025] DebtRatio
#   → IV médio-baixo, mas representa o comprometimento de renda — conceito
#     central em análise de crédito. Mantida apesar do problema de qualidade
#     identificado na EDA (winsorização contextual aplicada).
#
# [IV 0.025] NumberOfDependents
#   → IV limítrofe, mas mantida por relevância de negócio. Número de
#     dependentes afeta a capacidade de pagamento e já está incorporada
#     em renda_per_capita — o modelo decide se há sinal adicional.
#
# [IV 0.008] flag_missing_income
#   → IV fraco isoladamente, mas representa um segmento real do mercado
#     (autônomos, aposentados sem contracheque). Confirmado na EDA que
#     o missing não é aleatório — carrega informação de perfil do cliente.
#
# ──────────────────────────────────────────────────────────────────────────────
# FEATURES DESCARTADAS
# ──────────────────────────────────────────────────────────────────────────────
#
# [IV 0.012] NumberRealEstateLoansOrLines
#   → IV fraco. Altamente correlacionada com NumberOfOpenCreditLinesAndLoans
#     que já está no modelo. Redundância sem ganho preditivo adicional.
#
# [IV 0.057] teve_atraso_60dias (criada)
#   → Redundante com teve_atraso_90dias e teve_qualquer_atraso que já estão
#     no modelo. Adicionar a versão de 60 dias aumenta dimensionalidade
#     sem ganho marginal de informação.
#
# [IV 0.067] teve_atraso_30dias (criada)
#   → Mesma justificativa de teve_atraso_60dias. A janela de 30 dias já
#     está representada por NumberOfTime30-59DaysPastDueNotWorse e
#     teve_qualquer_atraso.
#
# [IV 0.049] proporcao_credito_imobiliario (criada)
#   → IV médio-baixo. Correlacionada com NumberOfOpenCreditLinesAndLoans
#     e NumberRealEstateLoansOrLines. Não agrega dimensão nova ao modelo.
#
# [IV 0.004] flag_missing_dependents
#   → IV desprezível. O padrão de missing em dependentes está capturado
#     por flag_missing_income (93% de sobreposição identificada na EDA).
#
# ══════════════════════════════════════════════════════════════════════════════

In [ ]:
# Decisão final de features para modelagem
# Critério: IV > 0.02 + sem redundância severa + disponível em produção

features_modelo = [
    # Features originais validadas
    "RevolvingUtilizationOfUnsecuredLines",  # IV 1.11 — mais forte do dataset
    "NumberOfTimes90DaysLate",               # IV 0.88 — histórico grave
    "NumberOfTime30-59DaysPastDueNotWorse",  # IV 0.76 — histórico leve
    "NumberOfTime60-89DaysPastDueNotWorse",  # IV 0.60 — histórico moderado
    "age",                                   # IV 0.26 — perfil demográfico
    "MonthlyIncome",                         # IV 0.07 — capacidade de pagamento
    "NumberOfOpenCreditLinesAndLoans",       # IV 0.07 — exposição ao crédito
    "DebtRatio",                             # IV 0.03 — comprometimento de renda
    "NumberOfDependents",                    # IV 0.02 — custo de vida
    "flag_missing_income",                   # IV 0.008 — padrão de missing informativo

    # Features criadas que agregam valor
    "renda_per_capita",                      # IV 0.096 — supera MonthlyIncome isolado
    "teve_atraso_90dias",                    # IV 0.84 — binarização do sinal mais forte
    "teve_qualquer_atraso",                  # IV 1.18 — agregado de comportamento
    "total_atrasos",                         # contagem agregada
    "score_risco",                           # índice composto para o modelo explorar
]

# Features descartadas e motivo
descartadas = {
    "NumberRealEstateLoansOrLines": "IV 0.012 — fraco, redundante com OpenCreditLines",
    "flag_missing_dependents": "IV 0.004 — fraco demais",
    "proporcao_credito_imobiliario": "IV 0.049 — médio mas correlacionado com features mantidas",
    "teve_atraso_30dias": "IV 0.67 — redundante com teve_atraso_90dias e total_atrasos",
    "teve_atraso_60dias": "IV 0.57 — redundante com teve_atraso_90dias e total_atrasos",
}

print("=== Features selecionadas para modelagem ===")
for f in features_modelo:
    print(f"  ✓ {f}")

print(f"\nTotal: {len(features_modelo)} features")

print("\n=== Features descartadas ===")
for f, motivo in descartadas.items():
    print(f"  ✗ {f} — {motivo}")

# Salvar dataset final para modelagem
X = df[features_modelo]
y = df["SeriousDlqin2yrs"]

X.to_csv("../data/processed/X_features.csv", index=False)
y.to_csv("../data/processed/y_target.csv", index=False)

print(f"\nX salvo: {X.shape}")
print(f"y salvo: {y.shape}")